# FCC-ee HEB RF impedance and wakes

Fit the longitudinal FM and HOM 1–3 to the 100 m CST source. Use the fully decayed longitudinal HOMs for the passive budget. Fit the transverse plane from all samples of the approximately 200 mm normal CST wake with the native IDDEFIX Gaussian wake potential. The filtered CST file is an optional cross-check.

Run in the HEB package directory, repository root, or Example directory. The first cell can point to CST files stored elsewhere. The longitudinal input is Wp_single800mhz_wake100.txt. The normal transverse input is W_single_rf_cavity_trans_3mm.txt. This analysis was executed with IDDEFIX 0.3.0, SciPy, NumPy, pandas and Matplotlib.

Run this notebook before HEB_V26_2_ExtK20_Full_Wake_RF_Budget_and_Wake.ipynb. The budget reads the per-cavity RF tables in data/rf_cavities/IDDEFIX_fullwake_CST/.

In [ ]:
from dataclasses import dataclass, replace
from hashlib import sha256
from importlib.metadata import version
from io import StringIO
import json
from pathlib import Path
import re

import iddefix
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.constants import c
from scipy.fft import dct, dst, next_fast_len, rfft
from scipy.integrate import trapezoid
from scipy.optimize import least_squares, minimize_scalar
from scipy.signal import CZT

DATA_DIR = None  # HEB package data/ directory, e.g. Path("/path/to/fcc_ee_booster_pywit_model/data")
LONGITUDINAL_WAKE_FILE = None  # Wp_single800mhz_wake100.txt
TRANSVERSE_WAKE_FILE = None   # W_single_rf_cavity_trans_3mm.txt
FILTERED_TRANSVERSE_WAKE_FILE = None  # optional W_single_rf_cavity_trans_filter_3mm.txt
RUN_FILTERED_CROSS_CHECK = True
TRANSVERSE_OFFSET_MM = 12.0
FILTERED_OFFSET_MM = 6.0

RF_CASE = "IDDEFIX_fullwake_CST"
FMAX_GHZ = 50.0
DISPLAY_STEP_MHZ = 10.0

cwd = Path.cwd().resolve()
if DATA_DIR is None:
    candidates = (
        path for base in (cwd, *cwd.parents)
        for path in (base / "data", base / "fcc_ee_booster_pywit_model/data", base)
    )
    DATA = next((path for path in candidates if (path / "rf_cavities").is_dir()), None)
else:
    DATA = Path(DATA_DIR).expanduser().resolve()
if DATA is None or not (DATA / "rf_cavities").is_dir():
    raise FileNotFoundError("Set DATA_DIR to the HEB data directory containing rf_cavities/.")

RF_DATA = DATA / "rf_cavities"
OUT = RF_DATA / RF_CASE
RESULTS = OUT / "longitudinal"
BUDGET_RF_DIR = OUT


def find_source(override, name, *, required=True):
    if override is not None:
        return Path(override).expanduser().resolve(strict=True)
    candidates = (RF_DATA / name, cwd / name, cwd / "upload" / name)
    path = next((path for path in candidates if path.is_file()), None)
    if path is None and required:
        raise FileNotFoundError(f"Place {name} in {RF_DATA} or set its path above.")
    return path


long_wake_path = find_source(LONGITUDINAL_WAKE_FILE, "Wp_single800mhz_wake100.txt")
normal_wake_path = find_source(TRANSVERSE_WAKE_FILE, "W_single_rf_cavity_trans_3mm.txt")
source_paths = {"longitudinal_100m": long_wake_path, "transverse_normal": normal_wake_path}
if RUN_FILTERED_CROSS_CHECK:
    filtered_wake_path = find_source(
        FILTERED_TRANSVERSE_WAKE_FILE, "W_single_rf_cavity_trans_filter_3mm.txt", required=False
    )
    if filtered_wake_path is not None:
        source_paths["transverse_filtered"] = filtered_wake_path


def file_hash(path):
    return sha256(path.read_bytes()).hexdigest()


source_hashes = {key: file_hash(path) for key, path in source_paths.items()}
long_wake_hash = source_hashes["longitudinal_100m"]
IDDEFIX_BACKEND = f"native IDDEFIX {version('iddefix')}"
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"{IDDEFIX_BACKEND} · SciPy {version('scipy')}")
for key, path in source_paths.items():
    print(f"{key}  {path.name}")
print(f"RF exports  {OUT}")

## 1. Longitudinal fundamental and higher-order modes

The 100 m CST wake uses a 10 mm rms bunch. Reconstruct its signed complex impedance with DCT-I and DST-I. Sample the local fit windows on the original CST grid with a chirp-Z transform and use the finite-wake IDDEFIX formula in the magnitude fit.

The loaded \(Q_L\) values are fixed from the previous model. For HOM 1, include the complex response of FM, HOM 2 and HOM 3 in the corrected frequency search. The parameter table uses \(R/Q=2R_s/Q_L\).

In [ ]:
# Prescribe Q_L and fit Rs and fr locally to |Z|.
FIT_METHOD = "magnitude"  # v14/v15 primary fit; intentionally no complex refit
N_RF_CAVITIES = 112
SIGMA_BUDGET_M = 0.004
FMAX_EXPORT_HZ = 50e9
PLOT_DF_HZ = 0.2e6  # oversamples finite-record Fourier integral, not new information
PLOT_FMAX_HZ = 3.55e9

MODES = (
    {"name": "FM",         "f_ref": 801.584e6, "Q": 1.10e7, "half": 20e6},
    {"name": "HM1_1p494", "f_ref": 1.494e9,   "Q": 3800.,  "half": 25e6},
    {"name": "HM2_2p368", "f_ref": 2.368e9,   "Q": 4900.,  "half": 30e6},
    {"name": "HM3_3p38",  "f_ref": 3.380e9,   "Q": 8200.,  "half": 35e6},
)


@dataclass
class CST:
    s: np.ndarray
    wake: np.ndarray
    ds: float
    sigma: float
    length: float

def read_cst(fn: Path) -> CST:
    with fn.open(encoding="utf8") as stream:
        header = stream.readline()
    props = dict((k, float(v)) for k, v in re.findall(
        r"([A-Za-z_]+)\s*=\s*([-+0-9.eE]+)", header
    ))
    if "sigma_z" not in props or "wakelength_long" not in props:
        raise ValueError("Could not parse sigma_z and wakelength_long from CST header")
    arr = np.loadtxt(fn, comments="#")
    s = arr[:, 0] * 1e-3
    wake = -arr[:, 1] * 1e12  # exactly the single CST->IDDEFIX sign conversion in v15
    ds = float(np.median(np.diff(s)))
    if not np.allclose(np.diff(s), ds, rtol=3e-6, atol=1e-10):
        raise ValueError("Non-uniform CST wake grid")
    sigma, length = props["sigma_z"] * 1e-3, props["wakelength_long"] * 1e-3
    if not (abs(sigma - 0.010) < 1e-9 and abs(length - 100.0) < 1e-5):
        raise ValueError("This script requires the v15 100 m / 10 mm CST dataset")
    return CST(s, wake, ds, sigma, length)

cst = read_cst(long_wake_path)
print(f"CST: {len(cst.s):,} samples, s = {cst.s[0]:.6f} ... {cst.s[-1]:.6f} m, ds = {cst.ds*1e3:.6f} mm")
print(f"Wake length {cst.length:g} m; sigma_z = {cst.sigma*1e3:g} mm")
assert cst.s[0] < 0 and cst.s[-1] > 99.9

In [ ]:
def dct_dst_z(cst: CST, fmax: float = 3.55e9):
    """v15 two-sided paired DCT-I/DST-I Gaussian deconvolution, signed."""
    M = next_fast_len(int(np.ceil(cst.s[-1] / cst.ds)) + 1)
    x = np.arange(M + 1) * cst.ds
    pos = np.interp(x, cst.s, cst.wake, left=0., right=0.)
    neg = np.interp(-x, cst.s, cst.wake, left=0., right=0.)
    dt = cst.ds / c
    zr = dt / 2 * (dct(pos, type=1) + dct(neg, type=1))
    zi = np.zeros_like(zr)
    zi[1:-1] = dt / 2 * (dst(neg[1:-1], type=1) - dst(pos[1:-1], type=1))
    k = min(M, int(np.floor(fmax * 2 * M * cst.ds / c)))
    f = np.arange(k + 1) * c / (2 * M * cst.ds)
    gauss = np.exp(-0.5 * (2 * np.pi * f * cst.sigma / c)**2)
    return f, (zr[:k+1] + 1j * zi[:k+1]) / gauss

def czt_z(cst: CST, f0: float, df: float, n: int):
    """Exact-frequency integral on the original two-sided CST sample grid.

    This oversamples the *existing* Fourier integral: it does not recover
    independent frequency resolution finer than approximately c/L.
    """
    f = f0 + np.arange(n, dtype=float) * df
    calc = CZT(n=len(cst.wake), m=n,
               w=np.exp(-2j * np.pi * df * cst.ds / c),
               a=np.exp(2j * np.pi * f0 * cst.ds / c))
    z = (cst.ds / c) * calc(cst.wake) * np.exp(-2j * np.pi * f * cst.s[0] / c)
    gauss = np.exp(-0.5 * (2 * np.pi * f * cst.sigma / c)**2)
    return f, z / gauss

f_dct, z_dct = dct_dst_z(cst, fmax=PLOT_FMAX_HZ)
_, z_czt_dct = czt_z(cst, 0., float(f_dct[1]-f_dct[0]), len(f_dct))
dct_error = np.linalg.norm(z_dct - z_czt_dct) / np.linalg.norm(z_dct)
print(f"DCT/DST vs CZT check: relative complex L2 error = {dct_error:.3e}")
assert dct_error < 1e-3

# The original CST magnitude file is optional.
native_path = long_wake_path.with_name('Z_single800mhz_wake100.txt')
if long_wake_path.name == 'Wp_single800mhz_wake100.txt' and native_path.is_file():
    native = np.loadtxt(native_path, comments='#')
    f_cst_export = native[:, 0] * 1e9
    f_export, z_native = czt_z(
        cst, float(f_cst_export[0]), float(np.median(np.diff(f_cst_export))), len(f_cst_export))
    np.testing.assert_allclose(f_export, f_cst_export, rtol=0, atol=2e3)
    useful = (f_cst_export >= 0.25e9) & (f_cst_export <= PLOT_FMAX_HZ) & (native[:, 1] > 0)
    native_error = np.median(np.abs(np.abs(z_native[useful]) - native[useful, 1]) / native[useful, 1])
    print(f'CST magnitude check  median relative difference {native_error:.3%}')

In [ ]:
def iddefix_finite(f, Rs, Q, fr, L):
    return iddefix.Impedances.Resonator_longitudinal_imp(
        np.asarray(f, dtype=float), Rs, Q, fr, wake_length=L)

def iddefix_full(f, Rs, Q, fr):
    return iddefix.Impedances.Resonator_longitudinal_imp(
        np.asarray(f, dtype=float), Rs, Q, fr, wake_length=None)

# Longitudinal fit using the 100 m CST wake.
def fit_one(cst: CST, mode: dict, method: str, *, fixed_modes=()) -> dict:
    """Original fixed-Q local fit, with the other modes included for HOM 1."""
    f0, half, q = mode["f_ref"], mode["half"], mode["Q"]
    df = 0.20e6
    ff, zz = czt_z(cst, f0-half, df, int(np.floor(2*half/df)) + 1)
    selected = abs(ff-f0) < half
    ff, zz = ff[selected], zz[selected]
    outer = abs(ff-f0) > (1-.18)*half

    other_z = np.zeros_like(zz)
    if fixed_modes:
        parameters = {i: [row["Rs_Ohm"], row["Q_fixed"], row["f_fit_hz"]]
                      for i, row in enumerate(fixed_modes)}
        other_z = iddefix.Impedances.n_Resonator_longitudinal_imp(
            ff, parameters, wake_length=cst.length)

    if method == "magnitude":
        baseline = float(np.median(abs(zz[outer])))
        y = np.maximum(abs(zz)-baseline, 0.)
    else:
        local = zz - other_z
        baseline = np.median(local[outer].real) + 1j*np.median(local[outer].imag)
        y = local - baseline

    target_norm = max(np.linalg.norm(y), 1e-20)
    other_abs = abs(other_z)
    use_combined_magnitude = bool(fixed_modes) and method == "magnitude"

    def evaluate(fr, details=False):
        unit = iddefix_finite(ff, 1., q, fr, cst.length)
        if use_combined_magnitude:
            # Scale Rs by the norm of the unit spectrum for stable minimization.
            rs_scale = target_norm / np.linalg.norm(unit)
            # Beyond this bound the triangle inequality rules out improvement
            # over the zero-amplitude residual.
            upper = 2*(1 + np.linalg.norm(other_z)/target_norm)

            def error(scaled_rs):
                predicted = abs(other_z + scaled_rs*rs_scale*unit) - other_abs
                return np.linalg.norm(predicted-y)/target_norm

            amplitude_fit = minimize_scalar(
                error, bounds=(0., upper), method="bounded", options={"xatol": 1e-8})
            rs = float(amplitude_fit.x*rs_scale)
            residual = float(amplitude_fit.fun)
        else:
            if method == "magnitude":
                unit = abs(unit)
            rs = max(0., float(np.vdot(unit, y).real/np.vdot(unit, unit).real))
            residual = np.linalg.norm(y-rs*unit)/target_norm
        return (residual, rs) if details else residual

    limit = (2 if fixed_modes else 1)*c/cst.length
    trial = np.linspace(f0-limit, f0+limit, 161)
    scores = np.array([evaluate(frequency) for frequency in trial])
    seed = trial[np.argmin(scores)]
    step = trial[1]-trial[0]
    lo, hi = max(f0-limit, seed-2*step), min(f0+limit, seed+2*step)
    fr = float(minimize_scalar(evaluate, bounds=(lo, hi), method="bounded",
                               options={"xatol": 5.}).x) if hi > lo else float(seed)
    relative, rs = evaluate(fr, details=True)
    return {
        "name": mode["name"], "fit_type": method, "Q_fixed": q,
        "f_fit_hz": fr, "Rs_Ohm": rs, "R_over_Q_Ohm": 2*rs/q,
        "relative_local_residual": float(relative),
        "at_frequency_boundary": bool(abs(fr-f0) >= .95*limit),
        "Rs_over_Q_Ohm": rs/q, "linewidth_MHz": fr/q/1e6,
        "frequency_search_min_Hz": f0-limit, "frequency_search_max_Hz": f0+limit,
        "observed_CST_magnitude_crest_Hz": float(ff[np.argmax(abs(zz))]),
        "fixed_modes_in_response": ", ".join(row["name"] for row in fixed_modes),
    }


fits = [fit_one(cst, mode, FIT_METHOD) for mode in MODES]
hom1_index = next(i for i, row in enumerate(fits) if row["name"] == "HM1_1p494")
hom1_original = fits[hom1_index]
other_modes = [row for i, row in enumerate(fits) if i != hom1_index]
hom1_updated = fit_one(cst, MODES[hom1_index], FIT_METHOD, fixed_modes=other_modes)
fits[hom1_index] = hom1_updated

print(f"HOM 1 original center {hom1_original['f_fit_hz']/1e9:.8f} GHz, "
      f"local residual {hom1_original['relative_local_residual']:.1%}, "
      f"frequency bound {hom1_original['at_frequency_boundary']}")
print(f"HOM 1 updated center {hom1_updated['f_fit_hz']/1e9:.8f} GHz, "
      f"local residual {hom1_updated['relative_local_residual']:.1%}, "
      f"frequency bound {hom1_updated['at_frequency_boundary']}")
print(f"CST magnitude crest in the HOM 1 window "
      f"{hom1_updated['observed_CST_magnitude_crest_Hz']/1e9:.8f} GHz")
print(f"HOM 1 search interval {hom1_updated['frequency_search_min_Hz']/1e9:.8f}–"
      f"{hom1_updated['frequency_search_max_Hz']/1e9:.8f} GHz")

hom_fits = [row for row in fits if row['name'] != 'FM']
assert len(hom_fits) == 3 and all(row['Rs_Ohm'] > 0 for row in fits)

mode_names = {'FM': 'FM', 'HM1_1p494': 'HOM 1', 'HM2_2p368': 'HOM 2', 'HM3_3p38': 'HOM 3'}
fm_hom_table = pd.DataFrame(fits)
fm_hom_table.insert(0, 'Mode', fm_hom_table['name'].map(mode_names))
display(pd.DataFrame({
    'Mode': fm_hom_table['Mode'],
    'f_r [GHz]': fm_hom_table['f_fit_hz'] / 1e9,
    'Fixed Q_L': fm_hom_table['Q_fixed'].astype(int),
    'R_s [Ω/cavity]': fm_hom_table['Rs_Ohm'].map(lambda value: f'{value:.6g}'),
    'R/Q [Ω/cavity]': fm_hom_table['R_over_Q_Ohm'],
    'Local magnitude residual': fm_hom_table['relative_local_residual'].map('{:.1%}'.format),
    'At frequency bound': fm_hom_table['at_frequency_boundary'],
}).round(6))

### Longitudinal fit overlays

In [ ]:
n_plot = int(np.floor(PLOT_FMAX_HZ / PLOT_DF_HZ)) + 1
f_plot, z_cst = czt_z(cst, 0., PLOT_DF_HZ, n_plot)
z_iddefix_finite = sum((iddefix_finite(f_plot, r['Rs_Ohm'], r['Q_fixed'],
                                        r['f_fit_hz'], cst.length) for r in fits),
                        np.zeros_like(f_plot, dtype=complex))
z_option1 = sum((iddefix_full(f_plot, r['Rs_Ohm'], r['Q_fixed'], r['f_fit_hz'])
                 for r in hom_fits), np.zeros_like(f_plot, dtype=complex))

LEGEND = (
    "Original CST (100 m finite wake)",
    "IDDEFIX fit (finite FM + 3 HOMs)",
    "Fully decayed HOM 1–3",
)
STYLES = ("-", "--", "-.")  # solid is used ONLY for the original CST data

def plot_impedance(f_hz, curves, title, save_name):
    fig, axs = plt.subplots(3, 1, figsize=(12.5, 9.0), sharex=True,
                            layout="constrained")
    funcs = (np.real, np.imag, np.abs)
    labels = (r"$\Re Z_{\parallel}$ [$\Omega$/cavity]",
              r"$\Im Z_{\parallel}$ [$\Omega$/cavity]",
              r"$|Z_{\parallel}|$ [$\Omega$/cavity]")
    for k, (ax, func, ylabel) in enumerate(zip(axs, funcs, labels)):
        vals = []
        for z, name, linestyle in zip(curves, LEGEND, STYLES):
            y = func(z)
            if k == 2: y = np.where(y > 0, y, np.nan)
            ax.plot(f_hz*1e-9, y, linestyle=linestyle, linewidth=1.25,
                    label=name, alpha=.88)
            vals.append(np.abs(y[np.isfinite(y)]))
        if k == 2:
            ax.set_yscale("log")
        else:
            vv = np.concatenate(vals)
            linthresh = max(1., float(np.nanpercentile(vv, 70))*.025)
            ax.set_yscale("symlog", linthresh=linthresh)
            ax.axhline(0, color="gray", ls=":", alpha=.5, lw=.7)
        ax.grid(True, alpha=.25, which="both")
        ax.set_ylabel(ylabel)
    axs[0].legend(fontsize=9, loc="best")
    axs[-1].set_xlabel("Frequency [GHz]")
    axs[0].set_title(title)
    fig.savefig(RESULTS/save_name, dpi=140, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    return RESULTS/save_name

curves = (z_cst, z_iddefix_finite, z_option1)
plots = {}
plots["full"] = plot_impedance(f_plot, curves,
    "Longitudinal FM and HOM 1–3 · 100 m CST wake",
    "RF_FM_HOM_fit_overview.png")

for mode in MODES:
    select = np.abs(f_plot - mode['f_ref']) <= mode['half']
    subset = tuple(z[select] for z in curves)
    plots[mode['name']] = plot_impedance(
        f_plot[select], subset,
        f"Longitudinal impedance · {mode_names[mode['name']]}",
        f"RF_FM_HOM_zoom_{mode['name']}.png")
print("Saved figures:")
for name, fp in plots.items(): print("  ", name, fp)

## 2. Transverse fit to the complete CST wake

Use the recorded Gaussian bunch length and every sample from the selected offset block. The four resonators use positive shunt impedances and a bounded fit for \(R_s\), \(Q\), and \(f_r\), with no frequency taper. The CST wake in V/pC is divided by the drive offset in metres for the fit in V/C/m. Fit parameters and exported impedance are per cavity.

In [ ]:
@dataclass(frozen=True)
class TransverseRecord:
    path: Path
    s_mm: np.ndarray
    wake_si: np.ndarray
    sigma_mm: float
    offset_mm: float

    @property
    def time(self):
        return self.s_mm * 1e-3 / c

    @property
    def dt(self):
        return (self.time[-1] - self.time[0]) / (len(self.s_mm) - 1)

    @property
    def sigma_t(self):
        return self.sigma_mm * 1e-3 / c

    @property
    def length_m(self):
        return float(self.s_mm[-1] * 1e-3)

    @property
    def impedance_factor(self):
        return 1j


def read_transverse_wake(path, offset_mm):
    contents = path.read_text(encoding="utf-8-sig")
    for block in re.split(r"(?m)(?=^#\s*Parameters\s*=)", contents):
        if not block.strip():
            continue
        header = block.splitlines()[0]
        props = {name: float(value) for name, value in re.findall(
            r"([A-Za-z_]+)\s*=\s*([-+0-9.eE]+)", header
        )}
        if not np.isclose(props.get("xbeam", np.nan), offset_mm):
            continue
        if not (offset_mm > 0 and np.isclose(props["xtest"], -offset_mm)
                and props["ybeam"] == props["ytest"] == 0 and props["sigma_z"] > 0):
            raise ValueError(f"{path.name} has an incompatible offset geometry or bunch length.")
        data = np.loadtxt(StringIO(block), ndmin=2)
        if data.shape[1] != 2 or len(data) < 3 or not np.isfinite(data).all():
            raise ValueError(f"{path.name} needs two finite columns with distance and wake.")
        s_mm, wake_v_pc = data.T
        ds = np.diff(s_mm)
        if not (s_mm[0] < 0 < s_mm[-1] and np.all(ds > 0)
                and np.allclose(ds, np.median(ds), atol=1e-9, rtol=1e-7)):
            raise ValueError(f"{path.name} needs a uniform, increasing distance grid about s = 0.")
        # Raw CST wake is V/pC for the recorded offset.
        wake_si = wake_v_pc * 1e12 / (offset_mm * 1e-3)
        return TransverseRecord(path, s_mm, wake_si, props["sigma_z"], offset_mm)
    raise ValueError(f"{path.name} has no drive offset {offset_mm:g} mm.")


transverse_records = {
    "Normal": read_transverse_wake(normal_wake_path, TRANSVERSE_OFFSET_MM),
}
if "transverse_filtered" in source_paths:
    transverse_records["Filtered"] = read_transverse_wake(
        source_paths["transverse_filtered"], FILTERED_OFFSET_MM
    )
elif RUN_FILTERED_CROSS_CHECK:
    print("Filtered CST file absent. Running the normal CST fit.")

# Initial guesses from the two earlier four-resonator solutions.
# The fit below updates all 12 parameters from the complete CST wake.
TRANSVERSE_SEEDS = (
    np.array([[16956.3, 5.047, 1.234575e9], [907.2, 1.5705, 2.250837e9],
              [841.6, 9.431, 4.040678e9], [217.4, 0.5593, 7.692423e9]]),
    np.array([[17255.0, 5.246, 1.232575e9], [918.0, 1.4059, 2.213913e9],
              [872.6, 10.483, 4.048972e9], [213.2, 0.5400, 7.758282e9]]),
)


def fit_complete_wake(record):
    t = record.time
    engine = iddefix.EvolutionaryAlgorithm(
        t, record.wake_si, N_resonators=4, plane="transverse", fitFunction="wake potential",
        sigma=record.sigma_t, wake_length=record.length_m, objectiveFunction="real",
        compute_uncertainties=False,
    )
    lower = np.tile((0.0, np.log10(0.54), 0.20), 4)
    upper = np.tile((100.0, 4.0, 50.0), 4)
    weights = np.ones_like(t)
    weights[[0, -1]] = np.sqrt(0.5)
    norm = np.linalg.norm(weights * record.wake_si)
    if norm <= 0:
        raise ValueError(f"Empty dipole signal in {record.path.name}.")

    def unpack(values):
        rs_scaled, log_q, fr_ghz = values.reshape(-1, 3).T
        return np.column_stack((1e4 * rs_scaled, 10**log_q, 1e9 * fr_ghz)).ravel()

    def residual(values):
        model = engine.fitFunction(t, unpack(values))
        return weights * (model - record.wake_si) / norm

    solutions = []
    for seed in TRANSVERSE_SEEDS:
        rs, q, fr = seed.T
        initial = np.column_stack((rs / 1e4, np.log10(q), fr / 1e9)).ravel()
        solutions.append(least_squares(
            residual, np.clip(initial, lower, upper), bounds=(lower, upper), x_scale="jac",
            max_nfev=240, ftol=1e-12, xtol=1e-12, gtol=1e-12,
        ))
    converged = [result for result in solutions if result.success]
    if not converged:
        raise RuntimeError(f"The four-resonator wake fit failed for {record.path.name}.")
    best = min(converged, key=lambda result: result.cost)
    params = unpack(best.x).reshape(-1, 3)
    order = np.argsort(params[:, 2])
    params = params[order]
    engine.minimizationParameters = params.ravel()
    wake = engine.get_wake_potential(time_data=t, sigma=record.sigma_t)
    return {
        "engine": engine, "params": params, "wake_si": wake,
        "wake_relative_l2": float(np.linalg.norm(best.fun)),
        "active_bound": best.active_mask.reshape(-1, 3).any(axis=1)[order],
    }


tx_fits = {label: fit_complete_wake(rec) for label, rec in transverse_records.items()}
tx_parameters = pd.DataFrame([
    {
        "CST_input": label, "resonator": i + 1, "f_r_GHz": fr / 1e9, "Q": q,
        "Rs_Ohm_per_m": rs, "R_over_Q_Ohm_per_m": 2 * rs / q,
        "decay_length_mm": 1e3 * c * q / (np.pi * fr),
        "active_bound": bool(fit["active_bound"][i]),
    }
    for label, fit in tx_fits.items()
    for i, (rs, q, fr) in enumerate(fit["params"])
])
display(tx_parameters.round(6))

### Finite and fully decayed impedance

Use the actual last CST sample as the IDDEFIX finite point-wake length. Evaluate the fully decayed point-wake impedance with wake_length=None. For the Gaussian bunch measurement, compute the DCT/DST of the modeled bunch wake on the measured sample grid. Dividing the finite bunch spectrum by \(G(f)\) is useful for comparing the window effect. The small \(G(f)\) at high frequency amplifies the wake endpoint and its sidelobes.

In [ ]:
def dct_dst_impedance(record, fmax_hz=FMAX_GHZ * 1e9):
    if not 0 < fmax_hz < 0.5 / record.dt:
        raise ValueError('Choose a frequency limit below the wake grid Nyquist frequency.')
    n = len(record.wake_si)
    intervals = next_fast_len(max(n, int(np.ceil(1 / (2 * record.dt * DISPLAY_STEP_MHZ * 1e6)))))
    padded = np.zeros(intervals + 1)
    padded[:n] = record.wake_si
    # DCT-I handles the first endpoint. The last measured sample is
    # interior to the zero-padded array, so halve its trapezoidal weight.
    padded[n - 1] *= 0.5
    df_hz = 1 / (2 * intervals * record.dt)
    frequency = np.arange(int(np.floor(fmax_hz / df_hz)) + 1) * df_hz
    real = 0.5 * record.dt * dct(padded, type=1)[:len(frequency)]
    imag = np.zeros_like(real)
    imag[1:] = -0.5 * record.dt * dst(padded[1:-1], type=1)[:len(frequency) - 1]
    phase = np.exp(-2j * np.pi * frequency * record.time[0])
    return frequency, record.impedance_factor * phase * (real + 1j * imag)



def gaussian(frequency_hz, record):
    return np.exp(-0.5 * (2 * np.pi * frequency_hz * record.sigma_t)**2)


def fft_impedance(t, wake, fmax=50e9):
    dt = (t[-1] - t[0]) / (len(t) - 1)
    intervals = next_fast_len(max(len(t), int(np.ceil(1 / (2 * dt * DISPLAY_STEP_MHZ * 1e6)))))
    nfft = 2 * intervals
    df = 1 / (nfft * dt)
    f = np.arange(int(np.floor(fmax / df)) + 1) * df
    samples = wake.copy()
    samples[[0, -1]] *= 0.5
    return f, 1j * dt * np.exp(-2j * np.pi * f * t[0]) * rfft(samples, n=nfft)[:len(f)]


def l2_error(reference, prediction):
    return float(np.linalg.norm(prediction - reference) / np.linalg.norm(reference))


f_comparison = np.linspace(0, FMAX_GHZ * 1e9, int(FMAX_GHZ * 1000) + 1)
tx_summary = {}
for label, rec in transverse_records.items():
    fit = tx_fits[label]
    engine = fit["engine"]
    f, z_bunch = dct_dst_impedance(rec)
    f_fft, z_fft = fft_impedance(rec.time, rec.wake_si)
    np.testing.assert_allclose(f_fft, f, rtol=5e-15, atol=1e-7)
    np.testing.assert_allclose(z_fft, z_bunch, rtol=1e-11, atol=1e-9)
    f_fit, z_fit_bunch = dct_dst_impedance(replace(rec, wake_si=fit["wake_si"]))
    np.testing.assert_array_equal(f, f_fit)
    form_factor = gaussian(f, rec)

    z_finite = engine.get_impedance(f_comparison, wake_length=engine.wake_length)
    z_full = engine.get_impedance(f_comparison, wake_length=None)
    finite_peak, full_peak = np.argmax(z_finite.real), np.argmax(z_full.real)
    np.testing.assert_allclose(z_full[0], 1j * np.sum(fit["params"][:, 0] / fit["params"][:, 1]),
                               rtol=1e-12)

    t_point = np.linspace(0, engine.wake_length / c, int(np.ceil(rec.length_m / 0.000025)) + 1)
    point = iddefix.Wakes.n_Resonator_transverse_wake(t_point, fit["params"].ravel())
    point_f, point_fft = fft_impedance(t_point, point)
    point_analytic = engine.get_impedance(point_f, wake_length=engine.wake_length)
    point_error = l2_error(point_analytic, point_fft)
    if point_error >= 1e-5:
        raise RuntimeError(f"Finite point wake FFT disagrees with IDDEFIX for {rec.path.name}.")
    band = f >= 0.03e9
    tx_summary[label] = {
        "wake_length_m": rec.length_m,
        "number_of_samples": len(rec.s_mm),
        "sigma_z_mm": rec.sigma_mm,
        "drive_offset_mm": rec.offset_mm,
        "wake_relative_l2": fit["wake_relative_l2"],
        "bunch_spectrum_relative_l2": l2_error(z_bunch[band], z_fit_bunch[band]),
        "finite_point_FFT_relative_l2": point_error,
        "finite_ReZ_peak_Ohm_per_m": float(z_finite[finite_peak].real),
        "finite_peak_GHz": float(f_comparison[finite_peak] / 1e9),
        "full_ReZ_peak_Ohm_per_m": float(z_full[full_peak].real),
        "full_peak_GHz": float(f_comparison[full_peak] / 1e9),
    }
    fit.update(
        f_hz=f, z_bunch=z_bunch, z_fit_bunch=z_fit_bunch, form_factor=form_factor,
        z_finite=z_finite, z_full=z_full,
    )

display(pd.DataFrame.from_dict(tx_summary, orient="index").round(7))

In [ ]:
STYLE = {
    "cst": {"color": "#8797a0", "linewidth": 2.7},
    "fit": {"color": "#b94f30", "linewidth": 1.3, "linestyle": "--"},
    "finite": {"color": "#745382", "linewidth": 1.35, "linestyle": ":"},
    "full": {"color": "#15638b", "linewidth": 1.55},
}

for label, rec in transverse_records.items():
    fit = tx_fits[label]
    data_window = fit["z_bunch"] / fit["form_factor"]
    model_window = fit["z_fit_bunch"] / fit["form_factor"]
    f_cst_ghz = fit["f_hz"] / 1e9
    f_model_ghz = f_comparison / 1e9
    length_label = f"{rec.length_m * 1e3:.3f} mm"
    fig, axes = plt.subplots(2, 2, figsize=(12.1, 7.5), layout="constrained")
    for ax, field, part, name in zip(axes[0], (np.real, np.imag), ("Re", "Im"), ("Real", "Imaginary")):
        ax.plot(f_cst_ghz, field(data_window) / 1e3, label="CST window", **STYLE["cst"])
        ax.plot(f_model_ghz, field(fit["z_finite"]) / 1e3,
                label="Finite IDDEFIX point wake", **STYLE["finite"])
        ax.plot(f_model_ghz, field(fit["z_full"]) / 1e3,
                label="Fully decayed IDDEFIX", **STYLE["full"])
        ax.set(title=f"{name} impedance", xlim=(0, 5),
               xlabel="Frequency [GHz]", ylabel=f"{part} Z⊥ [kΩ/m]")
    axes[0, 0].legend(frameon=False, fontsize=8.6)

    ax = axes[1, 0]
    ax.plot(f_cst_ghz, abs(data_window) / 1e3, label="CST window / G", **STYLE["cst"])
    ax.plot(f_cst_ghz, abs(model_window) / 1e3, label="IDDEFIX bunch wake window / G", **STYLE["fit"])
    ax.plot(f_model_ghz, abs(fit["z_full"]) / 1e3, label="Fully decayed IDDEFIX", **STYLE["full"])
    ax.set(title="Gaussian-divided spectrum", xlim=(0, FMAX_GHZ),
           yscale="log", xlabel="Frequency [GHz]", ylabel="|Z⊥| [kΩ/m]")
    ax.legend(frameon=False, fontsize=8.6)

    ax = axes[1, 1]
    ax.plot(rec.s_mm, rec.wake_si * 1e-12, label="CST data", **STYLE["cst"])
    ax.plot(rec.s_mm, fit["wake_si"] * 1e-12, label="IDDEFIX wake fit", **STYLE["fit"])
    ax.set(title=f"Complete CST bunch wake · σz = {rec.sigma_mm:g} mm",
           xlabel="Trailing distance [mm]", ylabel="W⊥ [mV/(pC·mm)]")
    ax.legend(frameon=False, fontsize=8.6)
    for ax in axes.flat:
        ax.grid(alpha=.24)
    fig.suptitle(f"Transverse RF dipole fit · {label} CST · {length_label}")
    path = OUT / f"RF_transverse_{label.lower()}_full_wake_fit.png"
    fig.savefig(path, dpi=160)
    display(fig)
    plt.close(fig)

## 3. Export matched RF impedance and wakes

Export the longitudinal HOM 1–3 model on an adaptive 0–50 GHz grid and the fully decayed four-resonator transverse model for the two dipole channels. Compute the point-charge, 0.4 mm and 4 mm wakes directly from the fitted resonators. In the RF files, longitudinal wakes use the CST sign and transverse wakes use V/(pC·mm). The manifest supplies the per-cavity tables and their units to the budget notebook.

The transverse fit uses the complete 3 mm CST bunch wake. The 0.4 mm wake and the fully decayed impedance represent predictions of the fitted resonators. The transverse wake sampled through 2 m extrapolates beyond the available CST record.

In [ ]:
if sha256(long_wake_path.read_bytes()).hexdigest() != long_wake_hash:
    raise RuntimeError('The 100 m longitudinal source changed. Run all cells again.')

# First reproduce the V26.2 10 MHz sampling grid.
f_coarse = np.linspace(0., FMAX_EXPORT_HZ, int(FMAX_EXPORT_HZ/10e6) + 1)

# Add fine resonance cores and logarithmic wings extending hundreds of linewidths.
# A narrow-only patch (e.g. +/-10 MHz) can pass its local grid-refinement test
# but still bias the total HOM loss factor by ~0.1% due to coarsely sampled tails.
patches = []
for row in hom_fits:
    width = row['f_fit_hz'] / row['Q_fixed']
    f_center = row['f_fit_hz']
    inner_widths, outer_widths = 25, 600
    central = f_center + np.linspace(-inner_widths*width,
                                     inner_widths*width,
                                     2*inner_widths*30+1)
    wing_offset = np.geomspace(inner_widths*width, outer_widths*width, 500)
    patches.extend([central, f_center-wing_offset, f_center+wing_offset])
f_adaptive = np.unique(np.concatenate([f_coarse, *patches]))

def hom_only(f_hz):
    return sum((iddefix_full(f_hz, row['Rs_Ohm'], row['Q_fixed'], row['f_fit_hz'])
                for row in hom_fits), np.zeros_like(f_hz, dtype=complex))

z_adaptive = hom_only(f_adaptive)
z_coarse = hom_only(f_coarse)
assert np.all(z_adaptive.real >= -1e-9)

def loss_factor_v_per_pc(f_hz, Z, sigma_z_m):
    weight = np.exp(-(2*np.pi*f_hz*sigma_z_m/c)**2)
    return float(2*trapezoid(np.real(Z)*weight, f_hz)*1e-12)

new_kappa = loss_factor_v_per_pc(f_adaptive, z_adaptive, SIGMA_BUDGET_M)

from scipy.special import erfc

Z_BUDGET_NAME = 'Z_single_rf_cavity_long_IDDEFIX_HOM_only_ADAPTIVE_0to50GHz.txt'
POINT_WAKE_NAME = 'W_single_rf_cavity_long_IDDEFIX_HOM_only_POINT_0to200mm.txt'
WAKE_SIGMAS_MM = (0.4, 4.0)
WAKE_S_MIN_MM, WAKE_S_MAX_MM, WAKE_DS_MM = -10., 200., 0.025

# SOURCE_PATHS in heb_gold_tl.py expects 3 columns, frequencies in GHz, per-cavity Ohm.
z_budget_path = BUDGET_RF_DIR / Z_BUDGET_NAME
np.savetxt(
    z_budget_path,
    np.column_stack([f_adaptive / 1e9, z_adaptive.real, z_adaptive.imag]),
    fmt='%.12e',
    header=('Freq(GHz) Re(Z)[Ohm/cavity] Im(Z)[Ohm/cavity]; IDDEFIX full HOM1+HOM2+HOM3; '
            '800MHz FM excluded; NO broadband remainder; adaptive frequency grid; provisional')
)

# Analytic causal point-charge wake from *the same three fitted full resonators*.
def point_wake_iddefix_v_per_c(s_m):
    ss = np.atleast_1d(np.asarray(s_m, dtype=float))
    out = np.zeros_like(ss)
    positive = ss >= 0  # at s=0, use the causal right-hand limit W(0+)
    if not np.any(positive):
        return out
    tt = ss[positive] / c
    for row in hom_fits:
        Rs, Q, wr = row['Rs_Ohm'], row['Q_fixed'], 2*np.pi*row['f_fit_hz']
        B = wr / (2*Q)
        C = np.sqrt(wr*wr - B*B)
        out[positive] += (Rs*wr/Q) * np.exp(-B*tt) * (
            np.cos(C*tt) - (B/C)*np.sin(C*tt)
        )
    return out

# Exact Gaussian-convolved *wake potential*, using one-sided causal point wake.
# This formula is equivalent to iddefix.Wakes.Resonator_longitudinal_wake_potential
# in the underdamped Q>0.5 regime. sigma_z_m is a length, NOT a time.
def bunch_wake_iddefix_v_per_c(s_m, sigma_z_m):
    ss = np.atleast_1d(np.asarray(s_m, dtype=float))
    if sigma_z_m <= 0:
        raise ValueError('Gaussian sigma_z must be positive')
    t = ss/c
    sig_t = sigma_z_m/c
    out = np.zeros_like(t)
    for row in hom_fits:
        Rs, Q, wr = row['Rs_Ohm'], row['Q_fixed'], 2*np.pi*row['f_fit_hz']
        B = wr / (2*Q)
        C = np.sqrt(wr*wr - B*B)
        lam = -B + 1j*C
        # E[ exp(lam*u) H(u) ] under a Gaussian centered at t.
        arg = -(t + sig_t*sig_t*lam)/(np.sqrt(2)*sig_t)
        out += (Rs*wr/(2*Q)) * np.real(
            (1 + 1j*B/C) * np.exp(lam*t + 0.5*(lam*sig_t)**2) * erfc(arg)
        )
    return out

# Signs: IDDEFIX uses W>0 at 0+ for a passive longitudinal resonator;
# source CST wake files use the opposite longitudinal sign, so export -W/1e12.
# Use exactly representable grid containing s=0 and endpoints -10, +200 mm.
n_left = int(np.rint(abs(WAKE_S_MIN_MM)/WAKE_DS_MM))
n_right = int(np.rint(WAKE_S_MAX_MM/WAKE_DS_MM))
s_wake_mm = np.arange(-n_left, n_right + 1, dtype=float)*WAKE_DS_MM
assert np.isclose(s_wake_mm[0],WAKE_S_MIN_MM) and np.isclose(s_wake_mm[-1],WAKE_S_MAX_MM)
assert np.any(s_wake_mm == 0)

point_mask = s_wake_mm >= 0
s_point_mm = s_wake_mm[point_mask]
point_wake_cst_v_per_pc = -1e-12*point_wake_iddefix_v_per_c(s_point_mm*1e-3)
point_wake_path = BUDGET_RF_DIR / POINT_WAKE_NAME
np.savetxt(point_wake_path,
           np.column_stack([s_point_mm, point_wake_cst_v_per_pc]), fmt='%.12e',
           header=('s / mm    W_long [V/pC]; IDDEFIX HOM-only POINT wake; '
                   'CST longitudinal sign; W(0)=W(0+); per cavity; no 800MHz fundamental'))

wake_paths = {}
wake_potentials = {}
for sigma_mm in WAKE_SIGMAS_MM:
    name = f'W_single_rf_cavity_long_IDDEFIX_HOM_only_sigma{sigma_mm:g}mm.txt'
    arr = -1e-12*bunch_wake_iddefix_v_per_c(s_wake_mm*1e-3, sigma_mm*1e-3)
    p = BUDGET_RF_DIR / name
    np.savetxt(p, np.column_stack([s_wake_mm, arr]), fmt='%.12e',
               header=(f's / mm    W_long [V/pC]; Gaussian sigma_z={sigma_mm:g}mm; '
                       'IDDEFIX full HOM-only wake potential, CST longitudinal sign; '
                       'per cavity; FM excluded'))
    wake_paths[sigma_mm] = p
    wake_potentials[sigma_mm] = arr

# Same data as in the original fit, saved alongside exported wake/impedance sources.
fit_data_path = BUDGET_RF_DIR / 'RF_IDDEFIX_FM_HOM_fitted_parameters.csv'
fm_hom_table.to_csv(fit_data_path, index=False)

print(f'Exported longitudinal RF files to {BUDGET_RF_DIR}')
for p in [z_budget_path, point_wake_path, *wake_paths.values(), fit_data_path]:
    print(' ', p.name, f'({p.stat().st_size/1024:.1f} KiB)')

# The 4 mm loss factor from the two independent domains must agree.
kappa_from_bunch_potential = float(
    bunch_wake_iddefix_v_per_c(np.array([0.]), np.sqrt(2)*SIGMA_BUDGET_M)[0] * 1e-12
)
kappa_from_frequency = new_kappa
kappa_relative_difference = abs(kappa_from_bunch_potential - kappa_from_frequency) / max(kappa_from_frequency, 1e-100)
print(f'κ(4 mm), frequency-domain adaptive Z: {kappa_from_frequency:.8f} V/pC per cavity')
print(f'κ(4 mm), wake-domain Gaussian convolution: {kappa_from_bunch_potential:.8f} V/pC per cavity')
print(f'Independent relative difference: {kappa_relative_difference:.3e}')
assert kappa_relative_difference < 5e-5

# Confirm source files have the expected units, columns, monotonicity, and values.
z_verify = np.loadtxt(z_budget_path)
assert z_verify.shape[1] == 3 and len(z_verify) == len(f_adaptive)
assert np.all(np.diff(z_verify[:,0]) > 0) and np.min(z_verify[:,1]) >= -1e-7
np.testing.assert_allclose(z_verify[:,1], z_adaptive.real, rtol=1e-9, atol=1e-9)
for sigma_mm, p in wake_paths.items():
    check = np.loadtxt(p)
    assert check.shape[1] == 2 and len(check) == len(s_wake_mm)
    assert np.all(np.diff(check[:,0]) > 0) and np.isfinite(check).all()
    assert np.allclose(check[:,1], wake_potentials[sigma_mm], rtol=1e-9, atol=1e-12)
print('PASS: 3-column adaptive impedance and 2-column wake files validated.')

longitudinal_manifest = {
    'model': 'Option 1: full HOM1+HOM2+HOM3 (no FM, no CST broadband)',
    'source_fit': 'v14/v15 fixed-Q local magnitude fit with native IDDEFIX resonators. HOM 1 uses a wider frequency search and the other three fitted modes.',
    'backend': IDDEFIX_BACKEND,
    'hom1_original_fit': hom1_original,
    'fit_parameters': fits,
    'n_cavities_in_ring': N_RF_CAVITIES,
    'source_file_normalization': 'SINGLE CAVITY: multiply by 112 only in the HEB model',
    'frequency_grid': 'adaptive with dense windows around all HOMs, 0..50 GHz',
    'wake_data': 'V/pC per cavity, CST longitudinal sign = -(IDDEFIX V/C)/1e12',
    'data_files': {
        'impedance_per_cavity': z_budget_path.name,
        'point_wake_per_cavity': point_wake_path.name,
        'gaussian_wake_potential_per_cavity': {f'{k:g}mm':p.name for k,p in wake_paths.items()},
        'fitted_parameters': fit_data_path.name,
    },
    'validation': {
        'kappa_4mm_Z_V_per_pC_per_cavity': kappa_from_frequency,
        'kappa_4mm_wake_V_per_pC_per_cavity': kappa_from_bunch_potential,
        'kappa_relative_difference': kappa_relative_difference,
    },
}

np.savetxt(
    RESULTS / 'Z_long_CST_DCT_DST_100m.txt',
    np.column_stack([f_dct / 1e9, z_dct.real, z_dct.imag]),
    fmt='%.12e',
    header='frequency_GHz ReZ_Ohm_per_cavity ImZ_Ohm_per_cavity',
)
np.savetxt(
    RESULTS / 'Z_long_IDDEFIX_FM_HOM_finite_100m.csv',
    np.column_stack([f_plot / 1e9, z_cst.real, z_cst.imag, z_iddefix_finite.real, z_iddefix_finite.imag]),
    fmt='%.12e', delimiter=',', comments='',
    header='frequency_GHz,Re_CST_Ohm_per_cavity,Im_CST_Ohm_per_cavity,Re_IDDEFIX_Ohm_per_cavity,Im_IDDEFIX_Ohm_per_cavity',
)
print(f'4 mm passive HOM loss factor across {N_RF_CAVITIES} cavities  {N_RF_CAVITIES * new_kappa:.4f} V/pC')

In [ ]:
for label, path in source_paths.items():
    if file_hash(path) != source_hashes[label]:
        raise RuntimeError(f"{path.name} changed during the fit. Rerun this notebook.")


def artifact(path):
    return {"file": path.relative_to(OUT).as_posix(), "sha256": file_hash(path)}


def source_record(key, path):
    return {
        "path": path.relative_to(DATA).as_posix() if path.is_relative_to(DATA) else str(path),
        "sha256": source_hashes[key],
    }


def factor_checks(f_hz, component, scale, get_wake, sigmas_mm=(4.0, 2.62)):
    checks = {}
    for sigma_mm in sigmas_mm:
        weight = np.exp(-(2 * np.pi * f_hz * sigma_mm * 1e-3 / c)**2)
        from_impedance = float(2 * scale * trapezoid(component * weight, f_hz))
        from_wake = float(get_wake(np.sqrt(2) * sigma_mm))
        error = abs(from_impedance - from_wake) / max(abs(from_wake), 1e-30)
        if error > 5e-5:
            raise ValueError(f"Wake and impedance factor mismatch at σz = {sigma_mm:g} mm: {error:.3e}")
        checks[f"{sigma_mm:g}"] = {
            "frequency_integral": from_impedance, "wake_convolution": from_wake, "relative_error": error,
        }
    return checks


long_checks = factor_checks(
    f_adaptive, z_adaptive.real, 1e-12,
    lambda sigma_mm: bunch_wake_iddefix_v_per_c(np.array([0.]), sigma_mm * 1e-3)[0] * 1e-12,
)
planes = {
    "long": {
        "iddefix_fit": {"impedance": artifact(z_budget_path), "wake": artifact(wake_paths[0.4])},
        "parameters": artifact(fit_data_path),
        "point_wake": artifact(point_wake_path),
        "wake_sigma4mm": artifact(wake_paths[4.0]),
        "hom1_frequency_Hz": hom1_updated["f_fit_hz"],
        "loss_factor_4mm_V_per_pC_per_cavity": new_kappa,
        "factor_checks_by_sigma_mm": long_checks,
    },
}

normal_fit = tx_fits["Normal"]
params = normal_fit["params"].ravel()
f_tx = np.linspace(0, FMAX_EXPORT_HZ, int(FMAX_EXPORT_HZ / 1e6) + 1)
z_tx = normal_fit["engine"].get_impedance(f_tx, wake_length=None)
assert np.isfinite(z_tx).all() and (z_tx.real >= 0).all()

# Preserve the dense short-range grid used by the RF wake exports.
s_tx = np.r_[s_wake_mm, np.arange(200.25, 2000.01, 0.25)]
t_tx = s_tx * 1e-3 / c
point_tx = iddefix.Wakes.n_Resonator_transverse_wake(t_tx, params) * 1e-15
np.testing.assert_array_equal(point_tx[s_tx <= 0], 0.0)


def tx_bunch(time_s, sigma_mm):
    return 1e-15 * iddefix.Wakes.n_Resonator_transverse_wake_potential(
        time_s, params, sigma=sigma_mm * 1e-3 / c
    )


bunch_tx = {sigma_mm: tx_bunch(t_tx, sigma_mm) for sigma_mm in (0.4, 4.0)}
tx_checks = factor_checks(
    f_tx, z_tx.imag, 1e-15,
    lambda sigma_mm: tx_bunch(np.array([0.]), sigma_mm)[0],
)
tx_all_params = OUT / "RF_IDDEFIX_full_wake_transverse_parameters.csv"
tx_parameters.to_csv(tx_all_params, index=False)

for axis in ("x", "y"):
    stem = f"dip{axis}"
    z_path = OUT / f"Z_single_rf_cavity_{stem}_IDDEFIX_fullwake_0to50GHz.txt"
    point_path = OUT / f"W_single_rf_cavity_{stem}_IDDEFIX_POINT_0to2000mm.txt"
    params_path = OUT / f"RF_IDDEFIX_{stem}_fitted_parameters.csv"
    tx_parameters.loc[tx_parameters["CST_input"] == "Normal"].assign(plane=axis).to_csv(
        params_path, index=False
    )
    np.savetxt(z_path, np.column_stack((f_tx / 1e9, z_tx.real, z_tx.imag)), fmt="%.14e",
               header="frequency_GHz Re_Z_Ohm_per_m Im_Z_Ohm_per_m · fully decayed IDDEFIX · per cavity")
    positive = s_tx >= 0
    np.savetxt(point_path, np.column_stack((s_tx[positive], point_tx[positive])), fmt="%.14e",
               header="s_mm W_V_per_pC_mm · causal point-charge wake · per cavity")
    w_paths = {}
    for sigma_mm, wake in bunch_tx.items():
        w_path = OUT / f"W_single_rf_cavity_{stem}_IDDEFIX_sigma{sigma_mm:g}mm.txt"
        np.savetxt(w_path, np.column_stack((s_tx, wake)), fmt="%.14e",
                   header=f"s_mm W_V_per_pC_mm · sigma_z = {sigma_mm:g} mm · per cavity")
        w_paths[sigma_mm] = w_path
    planes[axis] = {
        "iddefix_fit": {"impedance": artifact(z_path), "wake": artifact(w_paths[0.4])},
        "parameters": artifact(params_path),
        "point_wake": artifact(point_path),
        "wake_sigma4mm": artifact(w_paths[4.0]),
        "fitted_modes": len(normal_fit["params"]),
        "active_bound_hits": int(np.count_nonzero(normal_fit["active_bound"])),
        "factor_checks_by_sigma_mm": tx_checks,
        **tx_summary["Normal"],
    }

rf_manifest = {
    "schema_version": 1,
    "case": RF_CASE,
    "backend": IDDEFIX_BACKEND,
    "normalization": "per_cavity",
    "n_cavities": N_RF_CAVITIES,
    "frequency_band_GHz": [0, 50],
    "sigma_z_m": 0.4e-3,
    "impedance_columns": ["frequency_GHz", "Re_Z", "Im_Z"],
    "impedance_units": {"long": "Ohm", "x": "Ohm/m", "y": "Ohm/m"},
    "wake_columns": ["s_mm", "wake"],
    "wake_units": {"long": "V/pC", "x": "V/(pC mm)", "y": "V/(pC mm)"},
    "longitudinal_wake_sign": "CST",
    "longitudinal_model": "Fully decayed HOM 1–3 from the 100 m CST fit",
    "transverse_model": "Four resonators fitted to the complete normal CST bunch wake in the time domain",
    "transverse_frequency_taper": False,
    "transverse_export_wake_length_m": None,
    "transverse_wake_sample_end_mm": float(s_tx[-1]),
    "transverse_fit_checks": tx_summary,
    "transverse_fit_parameters": artifact(tx_all_params),
    "longitudinal_fit_details": longitudinal_manifest,
    "inputs": {key: source_record(key, path) for key, path in source_paths.items()},
    "planes": planes,
}

# Publish the handoff after checking the exported tables.
for plane, model in planes.items():
    for quantity, ncols in (("impedance", 3), ("wake", 2)):
        data = np.loadtxt(OUT / model["iddefix_fit"][quantity]["file"])
        if data.shape[1] != ncols or not np.isfinite(data).all() or np.any(np.diff(data[:, 0]) <= 0):
            raise RuntimeError(f"Invalid {plane} RF {quantity} table.")
        if quantity == "impedance":
            np.testing.assert_allclose(data[[0, -1], 0], [0, 50], rtol=0, atol=1e-9)

manifest_path = OUT / "RF_IDDEFIX_fullwake_CST_manifest.json"
pending = manifest_path.with_suffix(".json.tmp")
pending.write_text(json.dumps(rf_manifest, indent=2) + "\n", encoding="utf-8")
pending.replace(manifest_path)

display(pd.DataFrame([
    {"Plane": plane, "4 mm factor per cavity": model["factor_checks_by_sigma_mm"]["4"]["frequency_integral"],
     "Unit": rf_manifest["wake_units"][plane],
     "Independent relative check": model["factor_checks_by_sigma_mm"]["4"]["relative_error"]}
    for plane, model in planes.items()
]))
print(f"Budget handoff  {manifest_path}")